# Ejercicio 1 — Predice antes de correr

**Consigna:** antes de escribir una sola línea de código, respondé en la celda de texto de abajo:

> Si el dataset `reviews_hoteles.csv` tiene 85% de reseñas positivas y 15% negativas, ¿qué accuracy esperás que tenga un modelo que **siempre** responde "positivo", sin haber aprendido nada?

Escribí tu predicción con un número aproximado. Después vas a verificarla en el Ejercicio 4.

### Mi predicción: 85% (0.85) de accuracy.
Por qué: Como el accuracy mide el porcentaje de aciertos totales sobre el total de casos, si el 85% de las reseñas son realmente positivas y un modelo "tonto" responde siempre "positivo", va a acertar exactamente ese 85% de las veces y va a fallar en el 15% restante (las negativas). Esto demuestra que en datasets desbalanceados un accuracy alto puede ser engañoso si no se mira cómo predice la clase minoritaria.



# Ejercicio 2 — Pipeline completo con un dataset nuevo

**Consigna:** `reviews_hoteles.csv` tiene reseñas de hoteles (otro rubro, mismo problema: clasificar sentimiento). Armá el pipeline **completo** desde cero.

1. Cargar el dataset.
2. Split train/test.
3. Vectorizar con TF-IDF (`fit_transform` en train, `transform` en test — ¿te acordás por qué en ese orden?).
4. Entrenar un `LogisticRegression`.
5. Calcular accuracy.
6. Probar el modelo con 2 reseñas de hotel que inventes tu.

In [1]:
import pandas as pd

df_hoteles = pd.read_csv("reviews_hoteles.csv")


In [3]:
# TODO: split train/test
from sklearn.model_selection import train_test_split

# Definimos X (texto) e y (etiqueta) desde df_hoteles
X = df_hoteles[df_hoteles.columns[0]]
y = df_hoteles[df_hoteles.columns[-1]]

# Split train/test manteniendo la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Tamaño train:", len(X_train), "| Tamaño test:", len(X_test))

Tamaño train: 32 | Tamaño test: 8


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer()
X_train_tfidf = vectorizador.fit_transform(X_train)
X_test_tfidf = vectorizador.transform(X_test)


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Entrenar el modelo
modelo = LogisticRegression()
modelo.fit(X_train_tfidf, y_train)

# Calcular accuracy
y_pred = modelo.predict(X_test_tfidf)
print(f"Accuracy en test: {accuracy_score(y_test, y_pred):.2%}")

# Probar con 2 reseñas inventadas
nuevas_reseñas = [
    "Excelente ubicación, el desayuno muy completo y el personal súper amable.",
    "La habitación estaba sucia, el aire acondicionado no funcionaba y nadie nos dio solución."
]

predicciones = modelo.predict(vectorizador.transform(nuevas_reseñas))
for reseña, pred in zip(nuevas_reseñas, predicciones):
    print(f"Reseña: '{reseña}' --> Predicción: {pred}")


Accuracy en test: 37.50%
Reseña: 'Excelente ubicación, el desayuno muy completo y el personal súper amable.' --> Predicción: positivo
Reseña: 'La habitación estaba sucia, el aire acondicionado no funcionaba y nadie nos dio solución.' --> Predicción: negativo


In [6]:
y_pred = modelo.predict(X_test_tfidf)
print(f"Accuracy en test: {accuracy_score(y_test, y_pred):.2%}")


Accuracy en test: 37.50%


In [7]:
reseñas_nuevas = [
    "Excelente ubicación, el desayuno muy completo y el personal súper amable.",
    "La habitación estaba sucia, el aire acondicionado no funcionaba y nadie nos dio solución."
]

# Vectorizamos con transform (sin fit) usando el nombre de tu variable 'vectorizador'
predicciones = modelo.predict(vectorizador.transform(reseñas_nuevas))

for reseña, pred in zip(reseñas_nuevas, predicciones):
    print(f"Reseña: '{reseña}' --> Predicción: {pred}")


Reseña: 'Excelente ubicación, el desayuno muy completo y el personal súper amable.' --> Predicción: positivo
Reseña: 'La habitación estaba sucia, el aire acondicionado no funcionaba y nadie nos dio solución.' --> Predicción: negativo


## Ejercicio 3 — Duelo: tu modelo (Ejercicio 2) vs. pysentimiento

**Consigna:** vas a usar el modelo que corregiste en el Ejercicio 2 —entrenado sobre `reviews_base.csv`, sin data leakage— y compararlo contra pysentimiento en 10 reseñas con sarcasmo, negaciones y opiniones mixtas.

Usamos justamente el modelo del Ejercicio 2 y no el de hoteles del Ejercicio 3 porque `reviews_casos_limite.csv` es del mismo dominio (e-commerce) que `reviews_base.csv`. Comparar contra un modelo entrenado en otro rubro no sería una prueba justa.

1. Reentrená el pipeline corregido del Ejercicio 2 (podés copiar tu solución, o completar el esqueleto de acá abajo).
2. Instalá pysentimiento (`pip install pysentimiento`) y corré las mismas 10 reseñas.
3. Armá una tabla comparando: texto | predicción de tu modelo | predicción de pysentimiento | tu propio juicio como humano.
4. Marcá en qué casos discrepan los dos modelos.
5. Reflexión final (2-3 líneas): en los casos donde discreparon, ¿con cuál coincidís vos como humano? Pensá en el tamaño de cada dataset de entrenamiento — `reviews_base.csv` tiene 120 filas, pysentimiento fue entrenado con muchísimas más. ¿Cómo se relaciona eso con que uno de los dos acierte más seguido en casos ambiguos?



In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Cargamos las 10 reseñas de casos límite
df = pd.read_csv("reviews_casos_limite.csv")
display(df)

# 2. Reentrenamos el pipeline sobre reviews_base.csv (sin data leakage)
df_base = pd.read_csv("reviews_base.csv")
X_base = df_base[df_base.columns[0]]
y_base = df_base[df_base.columns[-1]]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42, stratify=y_base
)

vectorizer_base = TfidfVectorizer()
X_train_b_tfidf = vectorizer_base.fit_transform(X_train_b)

modelo_base = LogisticRegression()
modelo_base.fit(X_train_b_tfidf, y_train_b)

,texto,sentimiento
0,"Que buenísimo, llegó roto justo el día que lo ...",negativo
1,No está nada mal para el precio que pagué,positivo
2,"No es que sea malo, pero tampoco lo compraría ...",negativo
3,Increíble... tres semanas de espera para esto,negativo
4,El diseño me encanta pero la calidad deja bast...,negativo
5,No me arrepiento para nada de esta compra,positivo
6,"Justo lo que necesitaba, ni más ni menos",positivo
7,"Funciona, pero no esperen ninguna maravilla",negativo
8,"Nada que objetar, todo salió como esperaba",positivo
9,"Un producto que no defrauda ni sorprende, cump...",positivo


LogisticRegression()

In [9]:
# Vectorizamos los casos límite usando transform (nunca fit_transform)
textos_limite = df[df.columns[0]]
X_limite_tfidf = vectorizer_base.transform(textos_limite)

# Guardamos las predicciones de tu modelo en una columna del DataFrame
df["pred_mi_modelo"] = modelo_base.predict(X_limite_tfidf)
display(df)


,texto,sentimiento,pred_mi_modelo
0,"Que buenísimo, llegó roto justo el día que lo ...",negativo,positivo
1,No está nada mal para el precio que pagué,positivo,negativo
2,"No es que sea malo, pero tampoco lo compraría ...",negativo,negativo
3,Increíble... tres semanas de espera para esto,negativo,negativo
4,El diseño me encanta pero la calidad deja bast...,negativo,negativo
5,No me arrepiento para nada de esta compra,positivo,negativo
6,"Justo lo que necesitaba, ni más ni menos",positivo,positivo
7,"Funciona, pero no esperen ninguna maravilla",negativo,positivo
8,"Nada que objetar, todo salió como esperaba",positivo,positivo
9,"Un producto que no defrauda ni sorprende, cump...",positivo,negativo


In [10]:
!pip install pysentimiento -q
from pysentimiento import create_analyzer

analyzer = create_analyzer(task="sentiment", lang="es")

# Mapeamos las etiquetas de pysentimiento (POS, NEG, NEU) a español para comparar fácil
mapa_etiquetas = {"POS": "positivo", "NEG": "negativo", "NEU": "neutro"}

preds_pysentimiento = []
for texto in df[df.columns[0]]:
    resultado = analyzer.predict(texto)
    preds_pysentimiento.append(mapa_etiquetas.get(resultado.output, resultado.output))

df["pred_pysentimiento"] = preds_pysentimiento
df["discrepan"] = df["pred_mi_modelo"] != df["pred_pysentimiento"]

display(df)


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

,texto,sentimiento,pred_mi_modelo,pred_pysentimiento,discrepan
0,"Que buenísimo, llegó roto justo el día que lo ...",negativo,positivo,negativo,True
1,No está nada mal para el precio que pagué,positivo,negativo,positivo,True
2,"No es que sea malo, pero tampoco lo compraría ...",negativo,negativo,neutro,True
3,Increíble... tres semanas de espera para esto,negativo,negativo,negativo,False
4,El diseño me encanta pero la calidad deja bast...,negativo,negativo,negativo,False
5,No me arrepiento para nada de esta compra,positivo,negativo,positivo,True
6,"Justo lo que necesitaba, ni más ni menos",positivo,positivo,positivo,False
7,"Funciona, pero no esperen ninguna maravilla",negativo,positivo,neutro,True
8,"Nada que objetar, todo salió como esperaba",positivo,positivo,positivo,False
9,"Un producto que no defrauda ni sorprende, cump...",positivo,negativo,positivo,True


In [11]:
import pandas as pd

# Ajustamos el ancho de columna para leer el texto completo sin que se corte con "..."
pd.set_option('display.max_colwidth', None)

# Armamos la tabla comparativa final con las columnas sugeridas
tabla_comparativa = pd.DataFrame({
    "texto": df["texto"],
    "prediccion_propia": df["pred_mi_modelo"],
    "prediccion_pysentimiento": df["pred_pysentimiento"],
    "mi_juicio": df["sentimiento"],  # coincide con la etiqueta real de cada reseña
    "discrepan": df["discrepan"]
})

display(tabla_comparativa)


,texto,prediccion_propia,prediccion_pysentimiento,mi_juicio,discrepan
0,"Que buenísimo, llegó roto justo el día que lo necesitaba",positivo,negativo,negativo,True
1,No está nada mal para el precio que pagué,negativo,positivo,positivo,True
2,"No es que sea malo, pero tampoco lo compraría de nuevo",negativo,neutro,negativo,True
3,Increíble... tres semanas de espera para esto,negativo,negativo,negativo,False
4,El diseño me encanta pero la calidad deja bastante que desear,negativo,negativo,negativo,False
5,No me arrepiento para nada de esta compra,negativo,positivo,positivo,True
6,"Justo lo que necesitaba, ni más ni menos",positivo,positivo,positivo,False
7,"Funciona, pero no esperen ninguna maravilla",positivo,neutro,negativo,True
8,"Nada que objetar, todo salió como esperaba",positivo,positivo,positivo,False
9,"Un producto que no defrauda ni sorprende, cumple y ya",negativo,positivo,positivo,True


### Tu reflexión
En los casos donde discreparon (6 de las 10 reseñas), coincido como humano con pysentimiento, especialmente cuando hay sarcasmo (ej. "Que buenísimo, llegó roto...") o dobles negaciones (ej. "No está nada mal...", "No me arrepiento...", "no defrauda").

Esto se explica por el tamaño de entrenamiento y la arquitectura de cada modelo: mientras que nuestro modelo fue entrenado con apenas 120 filas mirando palabras aisladas (TF-IDF + Regresión Logística) y cae en la trampa de ver "buenísimo" como positivo o "mal" como negativo, pysentimiento fue preentrenado con millones de textos en español y comprende el contexto completo de la oración.